## Importando os dados

In [0]:
from pathlib import Path
import pandas as pd
import tarfile
import urllib.request

In [0]:
def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
    with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path="datasets")
    return pd.read_csv(Path("datasets/housing/housing.csv"))

housing = load_housing_data()

In [0]:
housing.head()

Colocando em portugues cada coluna:

(longitude, latitude, idade média do imovel, total de comodos, total de quartos, populacao, familia, renda_media, valor medio do imovel e proximidade do mar)

In [0]:
housing.info()

In [0]:
## descobrindo os valores categoricos de ocean_proximity
housing["ocean_proximity"].value_counts()

In [0]:
## describe básico das colunas
housing.describe()

In [0]:
## plotando um histograma para entender a natureza dos dados
import matplotlib.pyplot as plt

In [0]:
housing.hist(bins=50, figsize=(20,15))
plt.show()

Algumas variaveis conhecidas como tail-heavy, podem dificultar o desempenho de alguns algoritmos, deveremos tratar isso mais tarde

## Criando um conjunto de teste

In [0]:
import numpy as np

In [0]:
def split_train_test(data, test_ratio):
    shuffled_indices = np.random.permutation(len(data)) ## separando meus indices embaralhados 
    test_set_size = int(len(data) * test_ratio) ## define o tamanho do meu set de teste para meu test_ratio
    test_indices = shuffled_indices[:test_set_size] ## define que só vou pegar test_ratio valores do meu array embaralhado
    train_indices = shuffled_indices[test_set_size:] ## pego o resto do array, exceto, o test_indices
    ## data.iloc[train_indices] pega todas as linhas que foram separadas antes, ou seja, 80% do dataset
    ## data.iloc[test_set_size] pega todas as linhas que foram separadas para teste, ou seja, 20%
    return data.iloc[train_indices], data.iloc[test_indices] 


In [0]:
train_set, test_set = split_train_test(housing, 0.2)

In [0]:
len(train_set)

In [0]:
len(test_set)

In [0]:
from zlib import crc32

In [0]:
def test_set_check(identifier, test_ratio):
    # criando os hash e vendo se é menor que o test ratio
    # menor que test ratio = teste
    # maior que test_ratio = treino
    return crc32(np.int64(identifier)) & 0xffffffff < test_ratio * 2**32 

In [0]:
def split_train_test_by_id(data, test_ratio, id_column):
    ids = data[id_column] # transformando meu index em coluna
    in_test_set = ids.apply(lambda id_: test_set_check(id_, test_ratio)) # aplicando a funcao em todos os id
    return data.loc[~in_test_set], data.loc[in_test_set] # os valores boolenaos retornados pela funcao são colocados no df

In [0]:
housing_with_id = housing.reset_index() # adicionando um index

In [0]:
train_set, test_set = split_train_test_by_id(housing_with_id, 0.2, "index")

Utilizando outras caracteristicas para criar o ID's, além dos indices

In [0]:
housing_with_id.head()

In [0]:
housing_with_id["id"] = housing["longitude"] *1000 + housing["latitude"]

In [0]:
train_set, test_set = split_train_test_by_id(housing_with_id, 0.2, "id")

Podemos fazer isso com o sklearn também

Quando o dataset for pequeno, tome cuidado para não introduzir um viés de amostragem  

Amostragem estratificada, manter a proporção da amostra em relação a população 

### Amostragem estratificada

In [0]:
housing["income_cat"] = pd.cut(housing["median_income"],
                               bins=[0.,1.5,3.0,4.5, 6., np.inf],
                               labels=[1,2,3,4,5])
housing["income_cat"].hist()
plt.show()

In [0]:
from sklearn.model_selection import StratifiedShuffleSplit

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42) # criando minha amostra estratificada
# primeiro vamos pegar os indices referentes a teste e treino
# a função vai pegar o nosso dataframe e usar o gabarito(nesse caso o housing["income_cat"]) para fazer a amostra estratificada
for train_index, test_index in split.split(housing, housing["income_cat"]): 
  strat_train_set = housing.loc[train_index] # indice estratificada de treino, usamos o loc e não o iloc, pois o iloc pegaria os valores padrões de indice
  strat_test_set = housing.loc[test_index]
strat_test_set["income_cat"].value_counts() / len(strat_test_set)

Removendo os dados income_cat

In [0]:
for set_ in (strat_test_set, strat_train_set):
  set_.drop("income_cat", axis=1, inplace=True)

In [0]:
housing = strat_train_set.copy()

In [0]:
housing.plot(kind="scatter", x="longitude", y="latitude")

In [0]:
housing.plot(kind="scatter", x="longitude", y="latitude", alpha=0.1)

In [0]:
housing.plot(kind="scatter", x="longitude", y="latitude", alpha=0.4,
             s=housing["population"]/100, label="population", figsize=(10,7), # s define o tamanho do circulo s de size
             c="median_house_value", cmap=plt.get_cmap("jet"), colorbar=True) # c é de color, varia de acordo com o valor médio da casa
            # o plt.get_cmpa define a cor de azul para vermelho, chamando o jet
plt.legend()
plt.show()

In [0]:
corr_matrix = housing[
    housing.select_dtypes(include=['number']).columns
].corr()

In [0]:
corr_matrix["median_house_value"].sort_values(ascending=False)

In [0]:
from pandas.plotting import scatter_matrix

attributes =["median_house_value", "median_income", "total_rooms", "housing_median_age"]
scatter_matrix(housing[attributes], figsize=(12,8))

In [0]:
housing.plot(kind="scatter", x="median_income", y="median_house_value", alpha=0.1)

In [0]:
housing.head()

Trabalhando com atributos


In [0]:
housing["rooms_per_household"] = housing["total_rooms"]/housing["households"]
housing["bedrooms_per_room"] = housing["total_bedrooms"]/housing["total_rooms"]
housing["population_per_household"] = housing["population"]/housing["households"]

In [0]:
corr_matrix = housing[
    housing.select_dtypes(include=['number']).columns
].corr()
corr_matrix["median_house_value"].sort_values(ascending=False)

In [0]:
housing = strat_train_set.drop("median_house_value", axis=1) # tirando a coluna de respostas do modelo
housing_labels = strat_train_set["median_house_value"].copy() # mantendo a coluna de respostas

In [0]:
## temos tres opções para dados nulos
## 1 - dropar os dados
## 2 - preencher os dados
## 3 - drops o atributo 

from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")

housing_num = housing.drop("ocean_proximity", axis=1)

imputer.fit(housing_num)

In [0]:
imputer.statistics_

In [0]:
X = imputer.transform(housing_num)

In [0]:
housing_tr = pd.DataFrame(X, columns=housing_num.columns, index=housing_num.index)
housing_tr.head()

In [0]:
housing_tr.info()

In [0]:
housing_cat = housing[["ocean_proximity"]]

In [0]:
housing_cat.head(10)

In [0]:
from sklearn.preprocessing import OrdinalEncoder

ordinal_encoder = OrdinalEncoder()
housing_cat_encoded = ordinal_encoder.fit_transform(housing_cat)

housing_cat_encoded[:10]

In [0]:
ordinal_encoder.categories_

In [0]:
from sklearn.preprocessing import OneHotEncoder

cat_encoder = OneHotEncoder()

housing_cat_1hot = cat_encoder.fit_transform(housing_cat)
housing_cat_1hot

In [0]:
housing_cat_1hot.toarray()

In [0]:
cat_encoder.categories_

Estrutura de modelo básico

Explicação do código: 

BaseEstimator e TransformerMixin: São classes do scikit-learn que fornecem funcionalidades básicas para criar estimadores e transformadores personalizados. O TransformerMixin já implementa automaticamente o método fit_transform().

Índices: Define as posições das colunas no array de dados (provavelmente 3 = quartos, 4 = dormitórios, 5 = população, 6 = domicílios)



In [0]:
from sklearn.base import BaseEstimator, TransformerMixin

rooms_ix, bedrooms_ix, population_ix, households_ix = 3, 4, 5, 6 

def transform(self, X, y=None) -> Este método cria novas features combinadas:

    rooms_per_household: Total de quartos dividido pelo número de domicílios

    population_per_household: População dividida pelo número de quartos

    bedrooms_per_room (opcional): Número de dormitórios dividido pelo total de quartos

In [0]:
class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, add_bedrooms_per_room=True):
        self.add_bedrooms_per_room = add_bedrooms_per_room
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X, y=None):
        rooms_per_household = X[:, rooms_ix] / X[:, households_ix]
        population_per_household = X[:,population_ix] / X[:, rooms_ix]
        if self.add_bedrooms_per_room:
            bedrooms_per_room = X[:, bedrooms_ix] / X[:, rooms_ix]
            return np.c_[X, rooms_per_household, population_per_household, bedrooms_per_room]
        else:
            return np.c_[X, rooms_per_household, population_per_household]

In [0]:
attr_adder = CombinedAttributesAdder(add_bedrooms_per_room=False)
housing_extra_attibs = attr_adder.transform(housing.values)

## Pipeline

In [0]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

num_pipeline = Pipeline([
  ('imputer', SimpleImputer(strategy='median')),
  ('attribs_adder', CombinedAttributesAdder()),
  ('std_scaler', StandardScaler()),
])

In [0]:
housing_num_tr = num_pipeline.fit_transform(housing_num)

In [0]:
from sklearn.compose import ColumnTransformer
num_attribs = list(housing_num) # obtendo nomes das colunas numericas
cat_attribs = ["ocean_proximity"] # obtendo nomes das colunas categoricas

In [0]:
full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs), # tuplas, devem conter o nome atribuido (pode ser qualquer um), o transformador e a lista de nomes ou indices
    ("cat", OneHotEncoder(), cat_attribs)
])

In [0]:
housing_prepared = full_pipeline.fit_transform(housing)

## Treino do modelo

### Regressão linear


In [0]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(housing_prepared, housing_labels)

In [0]:
some_data = housing.iloc[:5]
some_labels = housing_labels.iloc[:5]
some_data_prepared = full_pipeline.transform(some_data)
print(f"Prediction {lin_reg.predict(some_data_prepared)}")
print(f"Labels {list(some_labels)}")

In [0]:
from sklearn.metrics import mean_squared_error

housing_predictions = lin_reg.predict(housing_prepared)
lin_mse = mean_squared_error(housing_labels, housing_predictions)
lin_rmse = np.sqrt(lin_mse)
lin_rmse

### Testando um modelo mais complexo

In [0]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = DecisionTreeRegressor()''
tree_reg.fit(housing_prepared, housing_labels)

In [0]:
housing_predictions = tree_reg.predict(housing_prepared)
tree_mse = mean_squared_error(housing_labels, housing_predictions)
tree_rmse = np.sqrt(tree_mse)
tree_rmse

### Usano o metodo K-folds

In [0]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(tree_reg, housing_prepared, housing_labels,
                         scoring="neg_mean_squared_error", cv=10)
tree_rmse_scores = np.sqrt(-scores) # usamos -, pois a validacao cruzada, é oposta ao MSE

In [0]:
def display_scores(scores): 
    print("Scores:", scores)
    print("Mean:", scores.mean())
    print("Standard deviation:", scores.std())

In [0]:
display_scores(tree_rmse_scores)